# Eniac A/B Test Analysis

## Homepage Button Experiment

This notebook analyses an A/B test for Eniac's homepage call-to-action button.

**Business question:** Which button version performs best, and should Eniac replace the original white **"SHOP NOW"** button?

The analysis compares four homepage button versions tested between **November 2 and November 16, 2021**.

| Version | Button Text | Colour | Type |
|---|---|---|---|
| A | SHOP NOW | White | Control |
| B | SHOP NOW | Red | Variant |
| C | SEE DEALS | White | Variant |
| D | SEE DEALS | Red | Variant |


## Data Sources and Limitations

The CSV files contain aggregated element-level click data, not individual visitor sessions.

Because of that, some experiment-level metrics are taken from the course case materials:

| Data | Source |
|---|---|
| Button clicks for available versions | CSV files |
| Total visitors per version | Course case materials |
| Correct Version C click count | Course case materials |
| Drop-off rate | Course case materials |
| Homepage-return rate | Course case materials |

Important data quality note: the Version C CSV may be duplicated or corrupted in some course materials. Therefore, the correct Version C button click count is manually set to **527**, based on the case information.


In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


In [ ]:
# File paths
# If you upload the CSV files to GitHub, keep them in the data/ folder:
#
# data/eniac_a.csv
# data/eniac_b.csv
# data/eniac_c.csv
# data/eniac_d.csv

DATA_PATHS = {
    "A": Path("../data/eniac_a.csv"),
    "B": Path("../data/eniac_b.csv"),
    "C": Path("../data/eniac_c.csv"),
    "D": Path("../data/eniac_d.csv"),
}

def load_version_data(paths):
    """Load the CSV files if they are available locally."""
    data = {}
    for version, path in paths.items():
        try:
            data[version] = pd.read_csv(path)
            print(f"Loaded Version {version}: {path}")
        except FileNotFoundError:
            data[version] = None
            print(f"File not found for Version {version}: {path}")
    return data

version_data = load_version_data(DATA_PATHS)


## 1. Data Quality Check

Before running the statistical test, the CSV files should be checked for basic integrity.

The key issue to verify is whether Version C contains the correct **"SEE DEALS"** button data or whether it is a duplicate of Version A.


In [ ]:
# Combine available CSV files for quick inspection

available_frames = []

for version, frame in version_data.items():
    if frame is not None:
        temp = frame.copy()
        temp["version"] = version
        available_frames.append(temp)

if available_frames:
    df = pd.concat(available_frames, ignore_index=True)
    print(f"Combined shape: {df.shape}")
    display(df.head())
else:
    df = pd.DataFrame()
    print("No CSV files loaded. The analysis below can still run using the case-provided experiment counts.")


In [ ]:
# Check whether Version A and Version C look identical

if version_data.get("A") is not None and version_data.get("C") is not None:
    a_values = version_data["A"].reset_index(drop=True)
    c_values = version_data["C"].reset_index(drop=True)

    same_shape = a_values.shape == c_values.shape
    identical = same_shape and a_values.equals(c_values)

    print(f"Version A and C same shape: {same_shape}")
    print(f"Version A and C identical: {identical}")

    if identical:
        print("Data quality note: Version C appears to be a duplicate of Version A.")
        print("The correct Version C click count will be taken from the case materials.")
else:
    print("Version A and/or C file not available for duplicate check.")


In [ ]:
# Helper function to extract button clicks from a CSV file

def get_button_clicks(frame, button_name, fallback_value):
    """Return the button click count from a CSV file, or fallback if unavailable."""
    if frame is None:
        return fallback_value

    if "Name" not in frame.columns or "No. clicks" not in frame.columns:
        return fallback_value

    matches = frame.loc[frame["Name"] == button_name, "No. clicks"]

    if matches.empty:
        return fallback_value

    return int(matches.iloc[0])


# Click counts
# A, B, and D can be extracted from the CSV files if available.
# C is manually set because the provided CSV may be corrupted/duplicated.

clicks = {
    "A": get_button_clicks(version_data.get("A"), "SHOP NOW", 512),
    "B": get_button_clicks(version_data.get("B"), "SHOP NOW", 281),
    "C": 527,
    "D": get_button_clicks(version_data.get("D"), "SEE DEALS", 193),
}

clicks


## 2. Contingency Table and Click-Through Rate

The chi-square test requires a contingency table showing clicks and no-clicks for each version.

Total visitor counts are provided by the case materials.


In [ ]:
# Total visitors per version from the case materials

visitors = {
    "A": 25326,
    "B": 24747,
    "C": 24876,
    "D": 25233,
}

no_clicks = {
    version: visitors[version] - clicks[version]
    for version in visitors
}

contingency_df = pd.DataFrame(
    {
        "Version_A": [clicks["A"], no_clicks["A"]],
        "Version_B": [clicks["B"], no_clicks["B"]],
        "Version_C": [clicks["C"], no_clicks["C"]],
        "Version_D": [clicks["D"], no_clicks["D"]],
    },
    index=["Click", "No-click"]
)

contingency_df


In [ ]:
# Calculate click-through rate per version

ctr_df = pd.DataFrame({
    "version": list(visitors.keys()),
    "button": ["White SHOP NOW", "Red SHOP NOW", "White SEE DEALS", "Red SEE DEALS"],
    "clicks": [clicks[v] for v in visitors],
    "visitors": [visitors[v] for v in visitors],
})

ctr_df["ctr"] = ctr_df["clicks"] / ctr_df["visitors"]
ctr_df["ctr_percent"] = ctr_df["ctr"] * 100

ctr_df.sort_values("ctr_percent", ascending=False)


In [ ]:
# Visualise CTR per version

plt.figure(figsize=(8, 5))

bars = plt.bar(
    ctr_df["version"],
    ctr_df["ctr_percent"],
    width=0.55
)

for bar, value in zip(bars, ctr_df["ctr_percent"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.03,
        f"{value:.2f}%",
        ha="center"
    )

plt.title("Click-through Rate by Button Version")
plt.xlabel("Version")
plt.ylabel("CTR (%)")
plt.ylim(0, max(ctr_df["ctr_percent"]) + 0.5)
plt.tight_layout()
plt.show()


## 3. Overall Chi-Square Test

The chi-square test checks whether the observed click-through differences across the four versions are statistically significant.

**Null hypothesis:** All button versions have the same click-through rate.

**Alternative hypothesis:** At least one button version has a different click-through rate.

Significance level: **α = 0.05**


In [ ]:
# Overall chi-square test

alpha = 0.05

chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_df)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.10f}")

if p_value < alpha:
    print("\nDecision: Reject the null hypothesis.")
    print("At least one button version performs significantly differently.")
else:
    print("\nDecision: Fail to reject the null hypothesis.")
    print("There is not enough evidence to conclude that the versions perform differently.")


In [ ]:
# Expected values under the null hypothesis

expected_df = pd.DataFrame(
    expected,
    index=contingency_df.index,
    columns=contingency_df.columns
)

expected_df


## 4. Post-Hoc Pairwise Tests

The overall chi-square test shows whether a difference exists, but not which specific versions differ.

To compare each pair of versions, pairwise chi-square tests are used.

Because there are six pairwise comparisons, a Bonferroni correction is applied:

**Adjusted α = 0.05 / 6 = 0.0083**


In [ ]:
# Pairwise chi-square tests with Bonferroni correction

versions = ["A", "B", "C", "D"]
alpha_adjusted = alpha / len(list(combinations(versions, 2)))

pairwise_results = []

for v1, v2 in combinations(versions, 2):
    pair_table = pd.DataFrame(
        {
            v1: [clicks[v1], no_clicks[v1]],
            v2: [clicks[v2], no_clicks[v2]],
        },
        index=["Click", "No-click"]
    )

    chi2_pair, p_pair, dof_pair, expected_pair = stats.chi2_contingency(pair_table)

    pairwise_results.append({
        "comparison": f"{v1} vs {v2}",
        "chi2": chi2_pair,
        "p_value": p_pair,
        "significant_after_bonferroni": p_pair < alpha_adjusted
    })

pairwise_df = pd.DataFrame(pairwise_results)
pairwise_df


## 5. Additional Business Metrics

Click-through rate alone does not fully determine the best version.

A button can receive more clicks but still perform worse if users abandon the process after clicking.

The case materials also provide:

- **Drop-off rate:** percentage of users who clicked but did not complete a purchase. Lower is better.
- **Homepage-return rate:** percentage of users who returned to the homepage after clicking. Lower is better.

Version B does not have complete data for these downstream metrics.


In [ ]:
# Additional metrics from the case materials

additional_metrics = pd.DataFrame({
    "version": ["A", "C", "D"],
    "button": ["White SHOP NOW", "White SEE DEALS", "Red SEE DEALS"],
    "drop_off_rate": [62.0, 71.0, 69.5],
    "homepage_return_rate": [5.3, 4.7, 2.6],
})

additional_metrics


In [ ]:
# Visualise additional metrics

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].bar(additional_metrics["version"], additional_metrics["drop_off_rate"], width=0.55)
axes[0].set_title("Drop-off Rate by Version")
axes[0].set_xlabel("Version")
axes[0].set_ylabel("Drop-off Rate (%)")
for i, value in enumerate(additional_metrics["drop_off_rate"]):
    axes[0].text(i, value + 0.5, f"{value:.1f}%", ha="center")

axes[1].bar(additional_metrics["version"], additional_metrics["homepage_return_rate"], width=0.55)
axes[1].set_title("Homepage-return Rate by Version")
axes[1].set_xlabel("Version")
axes[1].set_ylabel("Homepage-return Rate (%)")
for i, value in enumerate(additional_metrics["homepage_return_rate"]):
    axes[1].text(i, value + 0.05, f"{value:.1f}%", ha="center")

plt.tight_layout()
plt.show()


## 6. Weighted Score

Since CTR does not create a clear business winner between Version A and Version C, a weighted score is used to combine multiple metrics.

Weights:

- **CTR:** 50%
- **Drop-off rate:** 25%
- **Homepage-return rate:** 25%

CTR is better when higher. Drop-off rate and homepage-return rate are better when lower.

Version B is excluded from the weighted score because downstream metric data is unavailable.


In [ ]:
# Weighted score calculation

score_df = ctr_df[ctr_df["version"].isin(["A", "C", "D"])].copy()
score_df = score_df.merge(
    additional_metrics[["version", "drop_off_rate", "homepage_return_rate"]],
    on="version",
    how="left"
)

def normalize(series, higher_is_better=True):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series([0.5] * len(series), index=series.index)

    normalized = (series - min_value) / (max_value - min_value)

    if higher_is_better:
        return normalized

    return 1 - normalized

score_df["ctr_score"] = normalize(score_df["ctr_percent"], higher_is_better=True)
score_df["drop_off_score"] = normalize(score_df["drop_off_rate"], higher_is_better=False)
score_df["homepage_return_score"] = normalize(score_df["homepage_return_rate"], higher_is_better=False)

score_df["weighted_score"] = (
    score_df["ctr_score"] * 0.50
    + score_df["drop_off_score"] * 0.25
    + score_df["homepage_return_score"] * 0.25
)

score_df.sort_values("weighted_score", ascending=False)


In [ ]:
# Visualise weighted score ranking

ranking = score_df.sort_values("weighted_score", ascending=False)

plt.figure(figsize=(8, 5))
bars = plt.bar(ranking["version"], ranking["weighted_score"], width=0.55)

for bar, value in zip(bars, ranking["weighted_score"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{value:.3f}",
        ha="center"
    )

plt.title("Weighted Score by Version")
plt.xlabel("Version")
plt.ylabel("Weighted Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## 7. Final Recommendation

The overall chi-square test shows that the button versions do not perform equally.

However, the pairwise comparison between Version A and Version C is not statistically significant. This means CTR alone does not provide strong enough evidence to replace the original button with Version C.

Version B and Version D perform clearly worse and should not be used.

Although Version C has a slightly higher CTR, Version A performs better on the most important downstream metric: drop-off rate.

A lower drop-off rate means users who click Version A are more likely to continue toward purchase instead of abandoning the process.

## Final Decision

Eniac should keep the original white **"SHOP NOW"** button.

Future experiments should focus on clearer, product-specific call-to-action text, such as **"Shop iPhones"** or **"Shop Accessories"**, while avoiding major colour changes that may reduce engagement.


In [ ]:
# Final summary table

final_summary = pd.DataFrame({
    "Version": ["A - White SHOP NOW", "B - Red SHOP NOW", "C - White SEE DEALS", "D - Red SEE DEALS"],
    "CTR (%)": [
        ctr_df.loc[ctr_df["version"] == "A", "ctr_percent"].iloc[0],
        ctr_df.loc[ctr_df["version"] == "B", "ctr_percent"].iloc[0],
        ctr_df.loc[ctr_df["version"] == "C", "ctr_percent"].iloc[0],
        ctr_df.loc[ctr_df["version"] == "D", "ctr_percent"].iloc[0],
    ],
    "Drop-off Rate (%)": [62.0, np.nan, 71.0, 69.5],
    "Homepage-return Rate (%)": [5.3, np.nan, 4.7, 2.6],
})

final_summary
